# ExtraTrees tự code với tuning tham số và 10 feature đã chọn

Notebook này dùng các feature:

- `year`
- `is_covid`
- `log1p_gdp_per_capita`
- `sanitation`
- `electricity_water_interaction`
- `life_expectancy_lag1`
- `life_expectancy_trend_3y`
- `life_expectancy_volatility_3y`
- `log1p_gdp_per_capita_diff1`
- `sanitation_diff1`

Không dùng model sklearn, chỉ dùng các thư viện `numpy` và `pandas`.

Điểm khác so với Random Forest:
- ExtraTrees mặc định **không bootstrap** mà cho mỗi cây học trên toàn bộ dữ liệu train.
- Tại mỗi node, ExtraTrees chọn ngẫu nhiên một phần feature.
- Với mỗi feature được chọn, thuật toán sinh ngẫu nhiên threshold rồi chọn split tốt nhất trong các threshold ngẫu nhiên đó.


## Import và mô tả

In [1]:
# ============================================================
# EXTRA TREES TỰ CODE - DÙNG ĐÚNG 10 FEATURE ĐÃ CHỌN
# ============================================================
# Bài toán: dự đoán life_expectancy
#
# Ràng buộc:
#   - Không dùng model có sẵn của sklearn.
#   - Chỉ dùng numpy, pandas.
#
# Feature dùng để train ExtraTrees:
#   1. year
#   2. is_covid
#   3. log1p_gdp_per_capita
#   4. sanitation
#   5. electricity_water_interaction
#   6. life_expectancy_lag1
#   7. life_expectancy_trend_3y
#   8. life_expectancy_volatility_3y
#   9. log1p_gdp_per_capita_diff1
#   10. sanitation_diff1
#
# Lưu ý:
#   - country_code / country_name không được đưa trực tiếp vào model.
#   - country chỉ dùng tạm thời để tạo feature lịch sử và diff theo từng quốc gia.
#   - ExtraTrees dùng dữ liệu raw, không cần chuẩn hóa z-score.
#
# Cách chạy nếu file nằm trong prj/model/:
#   python extra_trees_selected_10_features.py

import json
import pickle
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. CONFIG

In [2]:
# ============================================================
# 1. CONFIG
# ============================================================

DATA_DIR = Path("..") / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_COL = "life_expectancy"
YEAR_COL = "year"
RANDOM_STATE = 42

# Các năm Covid. Có thể thử [2020, 2021] nếu muốn chỉ đánh dấu giai đoạn giảm mạnh.
COVID_YEARS = [2020, 2021, 2022]

# Nếu split dữ liệu theo thời gian thật:
#   train = năm cũ, val = năm sau train, test = năm sau val
# có thể đặt True để test dùng thêm lịch sử từ validation.
#
# Nếu split random theo dòng thì nên để False.
USE_VAL_HISTORY_FOR_TEST = False

COUNTRY_COL_CANDIDATES = [
    "country_code",
    "country_name",
    "country",
    "iso_code",
]

SELECTED_FEATURES = [
    "year",
    "is_covid",
    "log1p_gdp_per_capita",
    "sanitation",
    "electricity_water_interaction",
    "life_expectancy_lag1",
    "life_expectancy_trend_3y",
    "life_expectancy_volatility_3y",
    "log1p_gdp_per_capita_diff1",
    "sanitation_diff1",
]

# ============================================================
# CONFIG TUNING EXTRA TREES
# ============================================================
# Tuning sẽ chọn theo validation MAE, không chọn theo test MAE.
# Test chỉ dùng để đánh giá cuối cùng sau khi đã chọn tham số.

TUNING_N_ESTIMATORS = 80       # số cây dùng trong vòng lặp tuning để chạy nhanh
FINAL_N_ESTIMATORS = 500       # số cây train lại cho mô hình tốt nhất
TUNING_SEEDS = [RANDOM_STATE]  # muốn kiểm tra ổn định hơn: [42, 2024, 2025]

# Nếu chạy quá lâu, đặt MAX_TUNING_CONFIGS = 20 hoặc 30.
# Nếu muốn chạy toàn bộ grid, để None.
MAX_TUNING_CONFIGS = None

BASE_ET_CONFIG = {
    "n_estimators": TUNING_N_ESTIMATORS,
    "max_depth": 10,
    "min_samples_split": 4,
    "min_samples_leaf": 2,
    "max_features": 0.8,
    "bootstrap": False,
    "max_samples": 1.0,
    "n_random_thresholds": 1,
    "min_impurity_decrease": 0.0,
    "random_state": RANDOM_STATE,
    "verbose": 0,
}

# Grid mặc định gồm 48 cấu hình.
# Mục tiêu: kéo MAE xuống nhưng vẫn hạn chế overfit.
PARAM_GRID = {
    "max_depth": [6, 8, 10, 12],
    "min_samples_split": [4],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.6, 0.8],
    "n_random_thresholds": [1, 2],
    "bootstrap": [False],
    "max_samples": [1.0],
}

print("Current working directory:", Path.cwd())
print("Train:", TRAIN_PATH.resolve(), TRAIN_PATH.exists())
print("Val:", VAL_PATH.resolve(), VAL_PATH.exists())
print("Test:", TEST_PATH.resolve(), TEST_PATH.exists())
print("Số cấu hình tuning dự kiến:", np.prod([len(v) for v in PARAM_GRID.values()]))


Current working directory: c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Model
Train: C:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\train.csv True
Val: C:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\val.csv True
Test: C:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\test.csv True
Số cấu hình tuning dự kiến: 48


## 2. LOAD DATA

In [3]:
# ============================================================
# 2. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

if TARGET_COL not in train_df.columns:
    raise ValueError(f"Không tìm thấy target '{TARGET_COL}'. Các cột hiện có: {list(train_df.columns)}")

if YEAR_COL not in train_df.columns:
    raise ValueError(f"Không tìm thấy cột năm '{YEAR_COL}'. Các cột hiện có: {list(train_df.columns)}")

try:
    display(train_df.head())
except NameError:
    print(train_df.head())

Train shape: (3385, 16)
Val shape: (1128, 16)
Test shape: (1129, 16)


,country_name,country_code,year,population,pop_growth,life_expectancy,gdp_growth,sanitation,electricity,water_access,co2_emissions,labor_force,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap
0,Israel,ISR,2006.0,-0.203895,0.304521,80.553659,0.385675,1.146906,0.626003,0.741158,0.466569,0.117087,0,0.867560,0.759269,-0.934425
1,Qatar,QAT,2010.0,-0.245809,-0.486913,79.444000,2.744363,1.322539,0.626003,0.736250,4.256240,2.561888,0,1.644051,0.756579,-1.005932
2,Ghana,GHA,2022.0,-0.002721,0.399084,65.246000,0.082754,-1.289462,0.098067,0.039351,-0.442584,0.238940,0,-0.601728,-0.040621,0.482423
3,Tanzania,TZA,2008.0,0.069912,0.884395,59.896000,0.399970,-1.455111,-2.509724,-2.736538,-0.503541,2.642554,0,-1.359459,-2.267199,2.382605
4,Norway,NOR,2022.0,-0.216204,-0.234645,82.509756,-0.011425,0.903913,0.626003,0.741158,0.333926,0.471697,0,1.870898,0.759269,-0.833726


## 3. HÀM TÌM CỘT QUỐC GIA

In [4]:
# ============================================================
# 3. HÀM TÌM CỘT QUỐC GIA
# ============================================================

def find_country_col(df):
    for col in COUNTRY_COL_CANDIDATES:
        if col in df.columns:
            return col

    raise ValueError(
        "Không tìm thấy cột quốc gia. Cần một trong các cột: "
        + str(COUNTRY_COL_CANDIDATES)
    )


COUNTRY_COL = find_country_col(train_df)
print("Country column dùng để tạo lag/diff:", COUNTRY_COL)

Country column dùng để tạo lag/diff: country_code


## 4. FEATURE ENGINEERING CƠ BẢN

In [5]:
# ============================================================
# 4. FEATURE ENGINEERING CƠ BẢN
# ============================================================

def ensure_log1p_gdp_per_capita(df):
    """
    Đảm bảo có cột log1p_gdp_per_capita.

    Nếu chưa có nhưng có gdp_per_capita thì tạo:
        log1p_gdp_per_capita = log1p(gdp_per_capita)
    """

    output = df.copy()

    if "log1p_gdp_per_capita" not in output.columns:
        if "gdp_per_capita" not in output.columns:
            raise ValueError("Thiếu cả log1p_gdp_per_capita và gdp_per_capita.")
        output["log1p_gdp_per_capita"] = np.log1p(output["gdp_per_capita"].astype(float))

    return output


def add_is_covid(df, year_col=YEAR_COL, covid_years=COVID_YEARS):
    """
    is_covid = 1 nếu year thuộc COVID_YEARS, ngược lại = 0.

    is_covid được suy ra trực tiếp từ year, không dùng target.
    """

    output = df.copy()
    output["is_covid"] = output[year_col].isin(covid_years).astype(int)
    return output


def add_or_fix_electricity_water_interaction(df):
    """
    Tạo electricity_water_interaction nếu chưa có.

    Công thức:
        electricity_water_interaction = electricity * water_access / 100
    """

    output = df.copy()

    if "electricity_water_interaction" not in output.columns:
        required = ["electricity", "water_access"]
        missing = [col for col in required if col not in output.columns]

        if missing:
            raise ValueError(
                "Không tạo được electricity_water_interaction vì thiếu cột: "
                + str(missing)
            )

        output["electricity_water_interaction"] = (
            output["electricity"].astype(float)
            * output["water_access"].astype(float)
            / 100.0
        )

    return output


# Áp dụng feature cơ bản cho train/val/test
train_df = ensure_log1p_gdp_per_capita(train_df)
val_df = ensure_log1p_gdp_per_capita(val_df)
test_df = ensure_log1p_gdp_per_capita(test_df)

train_df = add_is_covid(train_df)
val_df = add_is_covid(val_df)
test_df = add_is_covid(test_df)

train_df = add_or_fix_electricity_water_interaction(train_df)
val_df = add_or_fix_electricity_water_interaction(val_df)
test_df = add_or_fix_electricity_water_interaction(test_df)

## 5. FEATURE LỊCH SỬ TUỔI THỌ

In [6]:
# ============================================================
# 5. FEATURE LỊCH SỬ TUỔI THỌ
# ============================================================

def add_life_expectancy_history_features(
    target_df,
    history_df,
    target_col=TARGET_COL,
    year_col=YEAR_COL,
    country_col=COUNTRY_COL,
    global_fill_value=None,
):
    """
    Tạo 3 feature lịch sử dùng trong model:
        - life_expectancy_lag1
        - life_expectancy_trend_3y
        - life_expectancy_volatility_3y

    Chống leakage:
        Với mỗi dòng cần dự đoán, chỉ dùng dữ liệu cùng quốc gia và year < year hiện tại.

    Xử lý năm đầu:
        - Không có lịch sử: lag1 = mean train, trend = 0, volatility = 0.
        - Có 1 năm lịch sử: lag1 = năm đó, trend = 0, volatility = 0.
        - Có 2 năm lịch sử: trend = chênh lệch năm gần nhất, volatility = std 2 năm.
        - Có >= 3 năm lịch sử: trend = slope tuyến tính 3 năm gần nhất, volatility = std 3 năm.
    """

    if global_fill_value is None:
        global_fill_value = history_df[target_col].mean()

    history = history_df[[country_col, year_col, target_col]].copy()
    history = history.dropna(subset=[country_col, year_col, target_col])
    history = history.sort_values([country_col, year_col])

    grouped_history = {
        country: group.sort_values(year_col)
        for country, group in history.groupby(country_col)
    }

    output = target_df.copy()

    lag1_values = []
    trend3_values = []
    volatility3_values = []

    for _, row in output.iterrows():
        country = row[country_col]
        year = row[year_col]

        if country not in grouped_history:
            past_values = np.array([], dtype=float)
        else:
            country_hist = grouped_history[country]
            past_values = (
                country_hist[country_hist[year_col] < year][target_col]
                .values
                .astype(float)
            )

        recent = past_values[-3:]

        if len(recent) == 0:
            lag1 = float(global_fill_value)
            trend3 = 0.0
            volatility3 = 0.0

        elif len(recent) == 1:
            lag1 = float(recent[-1])
            trend3 = 0.0
            volatility3 = 0.0

        elif len(recent) == 2:
            lag1 = float(recent[-1])
            trend3 = float(recent[-1] - recent[-2])
            volatility3 = float(recent.std())

        else:
            lag1 = float(recent[-1])

            x = np.array([0.0, 1.0, 2.0])
            y = recent.astype(float)
            trend3 = float(np.polyfit(x, y, 1)[0])

            volatility3 = float(recent.std())

        lag1_values.append(lag1)
        trend3_values.append(trend3)
        volatility3_values.append(volatility3)

    output["life_expectancy_lag1"] = lag1_values
    output["life_expectancy_trend_3y"] = trend3_values
    output["life_expectancy_volatility_3y"] = volatility3_values

    return output

## 6. FEATURE DIFF1

In [7]:
# ============================================================
# 6. FEATURE DIFF1
# ============================================================

def add_diff1_feature(
    target_df,
    history_df,
    feature,
    year_col=YEAR_COL,
    country_col=COUNTRY_COL,
):
    """
    Tạo:
        feature_diff1 = current_value - previous_available_value

    Chỉ dùng dữ liệu quá khứ cùng quốc gia từ history_df.
    Nếu không có lịch sử hoặc thiếu giá trị thì diff1 = 0.
    """

    output = target_df.copy()
    diff_col = f"{feature}_diff1"

    if feature not in output.columns:
        raise ValueError(f"Không tạo được {diff_col} vì thiếu cột gốc {feature}.")

    history = history_df.copy()

    grouped_history = {
        country: group.sort_values(year_col)
        for country, group in history.groupby(country_col)
    }

    diff_values = []

    for _, row in output.iterrows():
        country = row[country_col]
        year = row[year_col]
        current_value = row[feature]

        if pd.isna(current_value):
            diff_values.append(0.0)
            continue

        if country not in grouped_history:
            diff_values.append(0.0)
            continue

        country_hist = grouped_history[country]
        past = country_hist[country_hist[year_col] < year]

        if len(past) == 0 or feature not in past.columns:
            diff_values.append(0.0)
            continue

        previous_values = past[feature].dropna().values

        if len(previous_values) == 0:
            diff_values.append(0.0)
            continue

        previous_value = previous_values[-1]
        diff_values.append(float(current_value - previous_value))

    output[diff_col] = diff_values

    return output

## 7. TẠO FEATURE CHO TRAIN / VAL / TEST

In [8]:
# ============================================================
# 7. TẠO FEATURE CHO TRAIN / VAL / TEST
# ============================================================

global_train_life_mean = train_df[TARGET_COL].mean()

# Train: mỗi dòng train chỉ dùng lịch sử train có năm nhỏ hơn.
train_df_fe = add_life_expectancy_history_features(
    target_df=train_df,
    history_df=train_df,
    global_fill_value=global_train_life_mean,
)

# Val: chỉ dùng lịch sử từ train.
val_df_fe = add_life_expectancy_history_features(
    target_df=val_df,
    history_df=train_df,
    global_fill_value=global_train_life_mean,
)

# Test: mặc định chỉ dùng lịch sử từ train.
if USE_VAL_HISTORY_FOR_TEST:
    history_for_test = pd.concat([train_df, val_df], ignore_index=True)
else:
    history_for_test = train_df.copy()

test_df_fe = add_life_expectancy_history_features(
    target_df=test_df,
    history_df=history_for_test,
    global_fill_value=global_train_life_mean,
)

# Tạo diff1 cho GDP và sanitation.
for feature in ["log1p_gdp_per_capita", "sanitation"]:
    train_df_fe = add_diff1_feature(
        target_df=train_df_fe,
        history_df=train_df,
        feature=feature,
    )

    val_df_fe = add_diff1_feature(
        target_df=val_df_fe,
        history_df=train_df,
        feature=feature,
    )

    test_df_fe = add_diff1_feature(
        target_df=test_df_fe,
        history_df=history_for_test,
        feature=feature,
    )

print("Selected features:")
for feature in SELECTED_FEATURES:
    print("-", feature, "| exists:", feature in train_df_fe.columns)

Selected features:
- year | exists: True
- is_covid | exists: True
- log1p_gdp_per_capita | exists: True
- sanitation | exists: True
- electricity_water_interaction | exists: True
- life_expectancy_lag1 | exists: True
- life_expectancy_trend_3y | exists: True
- life_expectancy_volatility_3y | exists: True
- log1p_gdp_per_capita_diff1 | exists: True
- sanitation_diff1 | exists: True


## 8. CHUẨN BỊ X, y

In [9]:
# ============================================================
# 8. CHUẨN BỊ X, y
# ============================================================

missing_features = [col for col in SELECTED_FEATURES if col not in train_df_fe.columns]
if missing_features:
    raise ValueError("Thiếu selected features: " + str(missing_features))

X_train_df = train_df_fe[SELECTED_FEATURES].copy()
y_train = train_df_fe[TARGET_COL].values.astype(float)

X_val_df = val_df_fe[SELECTED_FEATURES].copy()
y_val = val_df_fe[TARGET_COL].values.astype(float)

X_test_df = test_df_fe[SELECTED_FEATURES].copy()
y_test = test_df_fe[TARGET_COL].values.astype(float)

print("X_train:", X_train_df.shape)
print("X_val:", X_val_df.shape)
print("X_test:", X_test_df.shape)

try:
    display(X_train_df.head())
except NameError:
    print(X_train_df.head())

X_train: (3385, 10)
X_val: (1128, 10)
X_test: (1129, 10)


,year,is_covid,log1p_gdp_per_capita,sanitation,electricity_water_interaction,life_expectancy_lag1,life_expectancy_trend_3y,life_expectancy_volatility_3y,log1p_gdp_per_capita_diff1,sanitation_diff1
0,2006.0,0,0.867560,1.146906,0.759269,80.151220,0.350000,0.328839,0.035986,0.024905
1,2010.0,0,1.644051,1.322539,0.756579,78.837000,0.548000,0.447621,0.145789,0.032296
2,2022.0,1,-0.601728,-1.289462,-0.040621,64.309000,0.393500,0.409766,0.009885,0.046940
3,2008.0,0,-1.359459,-1.455111,-2.267199,59.260000,1.665500,1.427000,0.138724,0.045985
4,2022.0,1,1.870898,0.903913,0.759269,83.163415,0.102439,0.109155,0.101940,0.002111


## 9. PREPROCESSOR CHO EXTRA TREES

In [10]:
# ============================================================
# 9. PREPROCESSOR CHO EXTRA TREES
# ============================================================

class NumericPreprocessor:
    """
    Preprocessor đơn giản cho ExtraTrees:
        - Chỉ xử lý numeric feature.
        - Fill missing bằng median của train.
        - Không scale vì mô hình cây không cần chuẩn hóa.
    """

    def __init__(self):
        self.feature_columns = None
        self.medians = None

    def fit(self, X_df):
        X = X_df.copy()
        self.feature_columns = X.columns.tolist()

        self.medians = X.median(numeric_only=True)
        self.medians = self.medians.fillna(0.0)

        return self

    def transform(self, X_df):
        X = X_df.copy()
        X = X.reindex(columns=self.feature_columns)
        X = X.fillna(self.medians)
        return X.values.astype(float)


preprocessor = NumericPreprocessor()
preprocessor.fit(X_train_df)

X_train = preprocessor.transform(X_train_df)
X_val = preprocessor.transform(X_val_df)
X_test = preprocessor.transform(X_test_df)

print("X_train array:", X_train.shape)
print("X_val array:", X_val.shape)
print("X_test array:", X_test.shape)


X_train array: (3385, 10)
X_val array: (1128, 10)
X_test array: (1129, 10)


## 10. METRICS

In [11]:
# ============================================================
# 10. METRICS
# ============================================================

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))


def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))


def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return 0.0

    return float(1 - ss_res / ss_tot)


def evaluate(split, y_true, y_pred):
    return {
        "model": "MyExtraTreesRegressor",
        "split": split,
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score_manual(y_true, y_pred),
    }


## 11. EXTRA TREE REGRESSOR

In [12]:
# ============================================================
# 11. EXTRA TREE REGRESSOR
# ============================================================

class TreeNode:
    def __init__(
        self,
        prediction=None,
        feature_index=None,
        threshold=None,
        left=None,
        right=None,
        is_leaf=True
    ):
        self.prediction = prediction
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.is_leaf = is_leaf


class MyExtraTreeRegressor:
    """
    Một cây hồi quy kiểu Extra Tree.

    Khác Decision Tree thường:
        - Mỗi node chỉ xét ngẫu nhiên một tập con feature.
        - Với mỗi feature được xét, threshold được sinh NGẪU NHIÊN trong khoảng [min, max].
        - Sau đó chọn split có SSE nhỏ nhất trong các split ngẫu nhiên hợp lệ.

    Tiêu chí split:
        Giảm SSE = parent_sse - (left_sse + right_sse)
    """

    def __init__(
        self,
        max_depth=12,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.8,
        n_random_thresholds=1,
        min_impurity_decrease=0.0,
        random_state=None
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_random_thresholds = n_random_thresholds
        self.min_impurity_decrease = min_impurity_decrease
        self.random_state = random_state

        self.root = None
        self.rng = np.random.default_rng(random_state)
        self.feature_importances_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        self.feature_importances_ = np.zeros(X.shape[1], dtype=float)
        self.root = self._build_tree(X, y, depth=0)

        total = self.feature_importances_.sum()
        if total > 0:
            self.feature_importances_ = self.feature_importances_ / total

        return self

    def _sse_from_sum(self, n, total_sum, total_sum2):
        if n <= 0:
            return 0.0
        return float(total_sum2 - (total_sum ** 2) / n)

    def _sse(self, y):
        if len(y) == 0:
            return 0.0
        return float(np.sum((y - np.mean(y)) ** 2))

    def _get_feature_indices(self, n_features):
        if self.max_features is None:
            size = n_features
        elif self.max_features == "sqrt":
            size = max(1, int(np.sqrt(n_features)))
        elif self.max_features == "log2":
            size = max(1, int(np.log2(n_features)))
        elif isinstance(self.max_features, int):
            size = min(n_features, self.max_features)
        elif isinstance(self.max_features, float):
            size = max(1, int(n_features * self.max_features))
        else:
            size = n_features

        return self.rng.choice(n_features, size=size, replace=False)

    def _find_best_split(self, X, y):
        n_samples, n_features = X.shape

        parent_sse = self._sse(y)

        best_feature = None
        best_threshold = None
        best_gain = 0.0
        best_score = np.inf

        feature_indices = self._get_feature_indices(n_features)

        for feature_idx in feature_indices:
            x_col = X[:, feature_idx]

            x_min = float(np.min(x_col))
            x_max = float(np.max(x_col))

            if x_min == x_max:
                continue

            # ExtraTrees: sinh threshold ngẫu nhiên thay vì duyệt toàn bộ ngưỡng tối ưu.
            thresholds = self.rng.uniform(
                low=x_min,
                high=x_max,
                size=max(1, int(self.n_random_thresholds))
            )
            thresholds = np.unique(thresholds)

            for threshold in thresholds:
                left_mask = x_col <= threshold
                n_left = int(np.sum(left_mask))
                n_right = n_samples - n_left

                if (
                    n_left < self.min_samples_leaf
                    or n_right < self.min_samples_leaf
                ):
                    continue

                y_left = y[left_mask]
                y_right = y[~left_mask]

                sum_left = float(np.sum(y_left))
                sum2_left = float(np.sum(y_left ** 2))

                sum_right = float(np.sum(y_right))
                sum2_right = float(np.sum(y_right ** 2))

                sse_left = self._sse_from_sum(n_left, sum_left, sum2_left)
                sse_right = self._sse_from_sum(n_right, sum_right, sum2_right)

                score = sse_left + sse_right
                gain = parent_sse - score

                if score < best_score and gain > best_gain:
                    best_score = score
                    best_gain = float(gain)
                    best_feature = int(feature_idx)
                    best_threshold = float(threshold)

        return best_feature, best_threshold, best_gain

    def _build_tree(self, X, y, depth):
        prediction = float(np.mean(y))

        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or np.var(y) < 1e-12
        ):
            return TreeNode(prediction=prediction, is_leaf=True)

        feature_idx, threshold, gain = self._find_best_split(X, y)
        gain_per_sample = gain / max(1, len(y))

        if (
            feature_idx is None
            or threshold is None
            or gain_per_sample < self.min_impurity_decrease
        ):
            return TreeNode(prediction=prediction, is_leaf=True)

        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask

        if (
            np.sum(left_mask) < self.min_samples_leaf
            or np.sum(right_mask) < self.min_samples_leaf
        ):
            return TreeNode(prediction=prediction, is_leaf=True)

        if self.feature_importances_ is not None:
            self.feature_importances_[feature_idx] += gain

        left_node = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_node = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return TreeNode(
            prediction=prediction,
            feature_index=feature_idx,
            threshold=threshold,
            left=left_node,
            right=right_node,
            is_leaf=False
        )

    def _predict_node_vectorized(self, X, node, indices, output):
        if node.is_leaf:
            output[indices] = node.prediction
            return

        selected_X = X[indices]
        left_mask = selected_X[:, node.feature_index] <= node.threshold

        left_indices = indices[left_mask]
        right_indices = indices[~left_mask]

        if len(left_indices) > 0:
            self._predict_node_vectorized(X, node.left, left_indices, output)

        if len(right_indices) > 0:
            self._predict_node_vectorized(X, node.right, right_indices, output)

    def predict(self, X):
        if self.root is None:
            raise RuntimeError("Tree chưa fit.")

        X = np.asarray(X, dtype=float)
        output = np.zeros(X.shape[0], dtype=float)
        indices = np.arange(X.shape[0])

        self._predict_node_vectorized(X, self.root, indices, output)
        return output


## 12. EXTRA TREES REGRESSOR TỰ CODE

In [13]:
# ============================================================
# 12. EXTRA TREES REGRESSOR TỰ CODE
# ============================================================

class MyExtraTreesRegressor:
    """
    ExtraTrees Regression tự code.

    Các thành phần chính:
        - Nhiều cây Extra Tree.
        - Mỗi cây thường học trên toàn bộ dữ liệu train, mặc định bootstrap=False.
        - Ở mỗi node: chọn ngẫu nhiên feature subset và sinh threshold ngẫu nhiên.
        - Ensemble averaging: dự đoán cuối là trung bình dự đoán của các cây.
    """

    def __init__(
        self,
        n_estimators=300,
        max_depth=12,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.8,
        bootstrap=False,
        max_samples=1.0,
        n_random_thresholds=1,
        min_impurity_decrease=0.0,
        random_state=42,
        verbose=1
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.max_samples = max_samples
        self.n_random_thresholds = n_random_thresholds
        self.min_impurity_decrease = min_impurity_decrease
        self.random_state = random_state
        self.verbose = verbose

        self.trees = []
        self.rng = np.random.default_rng(random_state)
        self.feature_importances_ = None

    def _get_sample_indices(self, n_samples):
        if self.max_samples is None:
            sample_size = n_samples
        elif isinstance(self.max_samples, float):
            sample_size = max(1, int(n_samples * self.max_samples))
            sample_size = min(n_samples, sample_size)
        else:
            sample_size = min(n_samples, int(self.max_samples))

        if self.bootstrap:
            return self.rng.choice(
                n_samples,
                size=sample_size,
                replace=True
            )

        # ExtraTrees chuẩn thường dùng toàn bộ dữ liệu khi bootstrap=False.
        # Nếu max_samples < 1.0 thì coi như random subspace theo dòng không lặp.
        if sample_size < n_samples:
            return self.rng.choice(
                n_samples,
                size=sample_size,
                replace=False
            )

        return np.arange(n_samples)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        n_samples = X.shape[0]
        self.trees = []

        for i in range(self.n_estimators):
            sample_indices = self._get_sample_indices(n_samples)

            X_sample = X[sample_indices]
            y_sample = y[sample_indices]

            tree = MyExtraTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=self.max_features,
                n_random_thresholds=self.n_random_thresholds,
                min_impurity_decrease=self.min_impurity_decrease,
                random_state=self.random_state + i
            )

            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

            if self.verbose and ((i + 1) % 10 == 0 or (i + 1) == self.n_estimators):
                print(f"Đã train {i + 1}/{self.n_estimators} cây")

        self._compute_feature_importances()
        return self

    def _compute_feature_importances(self):
        if len(self.trees) == 0:
            self.feature_importances_ = None
            return

        importances = [
            tree.feature_importances_
            for tree in self.trees
            if tree.feature_importances_ is not None
        ]

        if len(importances) == 0:
            self.feature_importances_ = None
            return

        self.feature_importances_ = np.mean(np.vstack(importances), axis=0)

        total = self.feature_importances_.sum()
        if total > 0:
            self.feature_importances_ = self.feature_importances_ / total

    def predict(self, X):
        if len(self.trees) == 0:
            raise RuntimeError("ExtraTrees chưa fit.")

        X = np.asarray(X, dtype=float)

        all_predictions = np.zeros((len(self.trees), X.shape[0]), dtype=float)

        for i, tree in enumerate(self.trees):
            all_predictions[i] = tree.predict(X)

        return np.mean(all_predictions, axis=0)


## 13. TUNING EXTRA TREES VÀ CHỌN BỘ THAM SỐ TỐI ƯU

In [14]:
# ============================================================
# 13. TUNING EXTRA TREES VÀ CHỌN BỘ THAM SỐ TỐI ƯU
# ============================================================

from itertools import product


def to_builtin_type(value):
    """Chuyển numpy scalar về Python scalar để lưu JSON/CSV ổn định."""
    if isinstance(value, np.generic):
        return value.item()
    return value


def build_param_combinations(param_grid):
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]

    combinations = []
    for combo in product(*values):
        combinations.append({
            key: to_builtin_type(value)
            for key, value in zip(keys, combo)
        })

    return combinations


def make_config(base_config, override_config, seed):
    config = dict(base_config)
    config.update(override_config)
    config["n_estimators"] = int(config["n_estimators"])
    config["random_state"] = int(seed)
    config["verbose"] = 0
    return config


def evaluate_one_config(config, config_id, seed):
    start_time = time.time()

    candidate_model = MyExtraTreesRegressor(**config)
    candidate_model.fit(X_train, y_train)

    train_time_seconds = time.time() - start_time

    pred_train_tmp = candidate_model.predict(X_train)
    pred_val_tmp = candidate_model.predict(X_val)

    train_metrics = evaluate("train", y_train, pred_train_tmp)
    val_metrics = evaluate("val", y_val, pred_val_tmp)

    row = {
        "config_id": config_id,
        "seed": seed,
        "train_time_seconds": train_time_seconds,

        "train_MAE": train_metrics["MAE"],
        "train_RMSE": train_metrics["RMSE"],
        "train_R2": train_metrics["R2"],

        "val_MAE": val_metrics["MAE"],
        "val_RMSE": val_metrics["RMSE"],
        "val_R2": val_metrics["R2"],

        "mae_gap_val_train": val_metrics["MAE"] - train_metrics["MAE"],
        "r2_gap_train_val": train_metrics["R2"] - val_metrics["R2"],
    }

    for key, value in config.items():
        row[key] = to_builtin_type(value)

    return row


param_combinations = build_param_combinations(PARAM_GRID)

if MAX_TUNING_CONFIGS is not None and MAX_TUNING_CONFIGS < len(param_combinations):
    rng_tuning = np.random.default_rng(RANDOM_STATE)
    selected_indices = rng_tuning.choice(
        len(param_combinations),
        size=MAX_TUNING_CONFIGS,
        replace=False
    )
    param_combinations = [param_combinations[i] for i in selected_indices]

print(f"Bắt đầu tuning {len(param_combinations)} cấu hình x {len(TUNING_SEEDS)} seed")
print("Tiêu chí chọn: validation MAE thấp nhất, có phạt nhẹ nếu overfit.")

all_tuning_rows = []

for config_id, override_config in enumerate(param_combinations, start=1):
    print("\n" + "=" * 80)
    print(f"Config {config_id}/{len(param_combinations)}:", override_config)

    for seed in TUNING_SEEDS:
        config = make_config(BASE_ET_CONFIG, override_config, seed)

        row = evaluate_one_config(
            config=config,
            config_id=config_id,
            seed=seed
        )

        all_tuning_rows.append(row)

        print(
            f"seed={seed} | "
            f"train_MAE={row['train_MAE']:.4f} | "
            f"val_MAE={row['val_MAE']:.4f} | "
            f"val_R2={row['val_R2']:.4f} | "
            f"gap_R2={row['r2_gap_train_val']:.4f}"
        )

tuning_runs_df = pd.DataFrame(all_tuning_rows)

# Các cột tham số cần group.
param_cols = [
    "n_estimators",
    "max_depth",
    "min_samples_split",
    "min_samples_leaf",
    "max_features",
    "bootstrap",
    "max_samples",
    "n_random_thresholds",
    "min_impurity_decrease",
]

agg_dict = {
    "train_time_seconds": ["mean", "sum"],
    "train_MAE": ["mean", "std"],
    "train_RMSE": ["mean", "std"],
    "train_R2": ["mean", "std"],
    "val_MAE": ["mean", "std"],
    "val_RMSE": ["mean", "std"],
    "val_R2": ["mean", "std"],
    "mae_gap_val_train": ["mean", "std"],
    "r2_gap_train_val": ["mean", "std"],
}

tuning_results_df = (
    tuning_runs_df
    .groupby(["config_id"] + param_cols, dropna=False)
    .agg(agg_dict)
    .reset_index()
)

# Làm phẳng tên cột sau groupby.
tuning_results_df.columns = [
    "_".join([str(part) for part in col if part != ""])
    if isinstance(col, tuple)
    else col
    for col in tuning_results_df.columns
]

# Nếu chỉ có 1 seed thì std = NaN, đổi thành 0 để dễ sort/lưu CSV.
std_cols = [col for col in tuning_results_df.columns if col.endswith("_std")]
tuning_results_df[std_cols] = tuning_results_df[std_cols].fillna(0.0)

# Điểm chọn:
# - val_MAE_mean là chính.
# - phạt nhẹ nếu R2 train cao hơn R2 val quá nhiều.
# - phạt nhẹ nếu val MAE cao hơn train MAE quá nhiều.
# - phạt nhẹ nếu kết quả không ổn định giữa các seed.
tuning_results_df["selection_score"] = (
    tuning_results_df["val_MAE_mean"]
    + 0.05 * np.maximum(0, tuning_results_df["r2_gap_train_val_mean"])
    + 0.02 * np.maximum(0, tuning_results_df["mae_gap_val_train_mean"])
    + 0.02 * tuning_results_df["val_MAE_std"]
)

tuning_results_df = tuning_results_df.sort_values(
    by=[
        "selection_score",
        "val_MAE_mean",
        "val_RMSE_mean",
        "r2_gap_train_val_mean",
    ],
    ascending=[True, True, True, True]
).reset_index(drop=True)

best_row = tuning_results_df.iloc[0].to_dict()

best_config = dict(BASE_ET_CONFIG)
for col in param_cols:
    best_config[col] = to_builtin_type(best_row[col])

# Train lại model tốt nhất với số cây lớn hơn để ổn định hơn.
best_config["n_estimators"] = FINAL_N_ESTIMATORS
best_config["random_state"] = RANDOM_STATE
best_config["verbose"] = 1

ET_CONFIG = best_config

print("\n" + "=" * 80)
print("TOP 10 CẤU HÌNH TỐT NHẤT THEO VALIDATION:")
display_cols = [
    "selection_score",
    "val_MAE_mean",
    "val_RMSE_mean",
    "val_R2_mean",
    "train_MAE_mean",
    "train_R2_mean",
    "mae_gap_val_train_mean",
    "r2_gap_train_val_mean",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split",
    "max_features",
    "n_random_thresholds",
    "bootstrap",
    "max_samples",
]

try:
    display(tuning_results_df[display_cols].head(10))
except NameError:
    print(tuning_results_df[display_cols].head(10))

print("\nBộ tham số tốt nhất được chọn:")
print(json.dumps(ET_CONFIG, ensure_ascii=False, indent=2))

tuning_runs_path = OUTPUT_DIR / "et_selected_10_features_tuning_runs.csv"
tuning_results_path = OUTPUT_DIR / "et_selected_10_features_tuning_results.csv"

tuning_runs_df.to_csv(tuning_runs_path, index=False)
tuning_results_df.to_csv(tuning_results_path, index=False)

print("\nĐã lưu kết quả tuning:")
print("-", tuning_runs_path)
print("-", tuning_results_path)

# ============================================================
# TRAIN LẠI MODEL CUỐI CÙNG VỚI BEST CONFIG
# ============================================================

print("\n" + "=" * 80)
print("Train lại mô hình cuối cùng với bộ tham số tốt nhất...")

model = MyExtraTreesRegressor(**ET_CONFIG)

start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"Hoàn tất training final model sau {train_time:.2f} giây")


Bắt đầu tuning 48 cấu hình x 1 seed
Tiêu chí chọn: validation MAE thấp nhất, có phạt nhẹ nếu overfit.

Config 1/48: {'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.6, 'n_random_thresholds': 1, 'bootstrap': False, 'max_samples': 1.0}
seed=42 | train_MAE=1.5258 | val_MAE=1.4708 | val_R2=0.9362 | gap_R2=-0.0063

Config 2/48: {'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.6, 'n_random_thresholds': 2, 'bootstrap': False, 'max_samples': 1.0}
seed=42 | train_MAE=1.1815 | val_MAE=1.1280 | val_R2=0.9549 | gap_R2=-0.0049

Config 3/48: {'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.8, 'n_random_thresholds': 1, 'bootstrap': False, 'max_samples': 1.0}
seed=42 | train_MAE=1.2735 | val_MAE=1.2158 | val_R2=0.9516 | gap_R2=-0.0072

Config 4/48: {'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.8, 'n_random_thresholds': 2, 'bootstrap': False, 'max_samples': 1.0}
seed=42

,selection_score,val_MAE_mean,val_RMSE_mean,val_R2_mean,train_MAE_mean,train_R2_mean,mae_gap_val_train_mean,r2_gap_train_val_mean,max_depth,min_samples_leaf,min_samples_split,max_features,n_random_thresholds,bootstrap,max_samples
0,0.593463,0.588095,1.277929,0.977086,0.353912,0.990773,0.234183,0.013687,12,1,4,0.8,2,False,1.0
1,0.620863,0.616166,1.338731,0.974854,0.410118,0.986377,0.206048,0.011523,12,2,4,0.8,2,False,1.0
2,0.624273,0.621364,1.287099,0.976756,0.497726,0.985483,0.123638,0.008727,10,1,4,0.8,2,False,1.0
3,0.626479,0.621268,1.298742,0.976334,0.394060,0.989663,0.227208,0.013329,12,1,4,0.6,2,False,1.0
4,0.638921,0.634586,1.303428,0.976163,0.446167,0.987487,0.188419,0.011325,12,1,4,0.8,1,False,1.0
5,0.645988,0.644226,1.391855,0.972819,0.560996,0.974769,0.083230,0.001950,12,4,4,0.8,2,False,1.0
6,0.647371,0.644533,1.364723,0.973868,0.522166,0.981676,0.122368,0.007808,10,2,4,0.8,2,False,1.0
7,0.648270,0.643969,1.351385,0.974376,0.454859,0.984761,0.189110,0.010384,12,2,4,0.6,2,False,1.0
8,0.654821,0.651447,1.356311,0.974189,0.503892,0.982651,0.147555,0.008462,12,2,4,0.8,1,False,1.0
9,0.673149,0.672123,1.431859,0.971234,0.623509,0.972305,0.048614,0.001071,10,4,4,0.8,2,False,1.0



Bộ tham số tốt nhất được chọn:
{
  "n_estimators": 500,
  "max_depth": 12,
  "min_samples_split": 4,
  "min_samples_leaf": 1,
  "max_features": 0.8,
  "bootstrap": false,
  "max_samples": 1.0,
  "n_random_thresholds": 2,
  "min_impurity_decrease": 0.0,
  "random_state": 42,
  "verbose": 1
}

Đã lưu kết quả tuning:
- model_outputs\et_selected_10_features_tuning_runs.csv
- model_outputs\et_selected_10_features_tuning_results.csv

Train lại mô hình cuối cùng với bộ tham số tốt nhất...
Đã train 10/500 cây
Đã train 20/500 cây
Đã train 30/500 cây
Đã train 40/500 cây
Đã train 50/500 cây
Đã train 60/500 cây
Đã train 70/500 cây
Đã train 80/500 cây
Đã train 90/500 cây
Đã train 100/500 cây
Đã train 110/500 cây
Đã train 120/500 cây
Đã train 130/500 cây
Đã train 140/500 cây
Đã train 150/500 cây
Đã train 160/500 cây
Đã train 170/500 cây
Đã train 180/500 cây
Đã train 190/500 cây
Đã train 200/500 cây
Đã train 210/500 cây
Đã train 220/500 cây
Đã train 230/500 cây
Đã train 240/500 cây
Đã train 250/500 

## 14. EVALUATE

In [15]:
# ============================================================
# 14. EVALUATE
# ============================================================

pred_train = model.predict(X_train)
pred_val = model.predict(X_val)
pred_test = model.predict(X_test)

results_df = pd.DataFrame([
    evaluate("train", y_train, pred_train),
    evaluate("val", y_val, pred_val),
    evaluate("test", y_test, pred_test),
])

print("Kết quả ExtraTrees:")

try:
    display(results_df)
except NameError:
    print(results_df)


Kết quả ExtraTrees:


,model,split,MAE,RMSE,R2
0,MyExtraTreesRegressor,train,0.350776,0.832505,0.991084
1,MyExtraTreesRegressor,val,0.584374,1.273183,0.977256
2,MyExtraTreesRegressor,test,0.544360,1.121882,0.983027


## 15. FEATURE IMPORTANCE

In [16]:
# ============================================================
# 15. FEATURE IMPORTANCE
# ============================================================

feature_importance_df = pd.DataFrame({
    "feature": preprocessor.feature_columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Feature importance:")

try:
    display(feature_importance_df)
except NameError:
    print(feature_importance_df)

Feature importance:


,feature,importance
0,electricity_water_interaction,0.437789
1,life_expectancy_lag1,0.401230
2,log1p_gdp_per_capita,0.092880
3,sanitation,0.041221
4,year,0.010844
5,life_expectancy_volatility_3y,0.005311
6,life_expectancy_trend_3y,0.004588
7,log1p_gdp_per_capita_diff1,0.002340
8,sanitation_diff1,0.002285
9,is_covid,0.001512


## 16. LƯU KẾT QUẢ, MODEL VÀ KẾT QUẢ TUNING

In [17]:
# ============================================================
# 16. LƯU KẾT QUẢ, MODEL VÀ KẾT QUẢ TUNING
# ============================================================

test_predictions_df = test_df_fe.copy()
test_predictions_df["prediction"] = pred_test
test_predictions_df["error"] = test_predictions_df[TARGET_COL] - test_predictions_df["prediction"]
test_predictions_df["abs_error"] = np.abs(test_predictions_df["error"])

results_path = OUTPUT_DIR / "et_selected_10_features_results.csv"
pred_path = OUTPUT_DIR / "et_selected_10_features_test_predictions.csv"
importance_path = OUTPUT_DIR / "et_selected_10_features_importance.csv"
config_path = OUTPUT_DIR / "et_selected_10_features_config.json"
model_path = OUTPUT_DIR / "et_selected_10_features_model.pkl"
preprocessor_path = OUTPUT_DIR / "et_selected_10_features_preprocessor.pkl"
best_config_path = OUTPUT_DIR / "et_selected_10_features_best_config.json"

results_df.to_csv(results_path, index=False)
test_predictions_df.to_csv(pred_path, index=False)
feature_importance_df.to_csv(importance_path, index=False)

with open(model_path, "wb") as f:
    pickle.dump(model, f)

with open(preprocessor_path, "wb") as f:
    pickle.dump(preprocessor, f)

config_to_save = {
    "model": "MyExtraTreesRegressor",
    "selection_rule": "Chọn theo validation MAE, không dùng test để chọn tham số",
    "target_col": TARGET_COL,
    "selected_features": SELECTED_FEATURES,
    "covid_years": COVID_YEARS,
    "use_val_history_for_test": USE_VAL_HISTORY_FOR_TEST,
    "country_col": COUNTRY_COL,
    "config": ET_CONFIG,
    "train_time_seconds": train_time,
    "output_files": {
        "metrics": str(results_path),
        "test_predictions": str(pred_path),
        "feature_importance": str(importance_path),
        "tuning_runs": str(OUTPUT_DIR / "et_selected_10_features_tuning_runs.csv"),
        "tuning_results": str(OUTPUT_DIR / "et_selected_10_features_tuning_results.csv"),
    }
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_to_save, f, ensure_ascii=False, indent=2)

with open(best_config_path, "w", encoding="utf-8") as f:
    json.dump(ET_CONFIG, f, ensure_ascii=False, indent=2)

print("Đã lưu:")
print("-", results_path)
print("-", pred_path)
print("-", importance_path)
print("-", config_path)
print("-", best_config_path)
print("-", model_path)
print("-", preprocessor_path)

if "tuning_results_df" in globals():
    print("-", OUTPUT_DIR / "et_selected_10_features_tuning_results.csv")

if "tuning_runs_df" in globals():
    print("-", OUTPUT_DIR / "et_selected_10_features_tuning_runs.csv")


Đã lưu:
- model_outputs\et_selected_10_features_results.csv
- model_outputs\et_selected_10_features_test_predictions.csv
- model_outputs\et_selected_10_features_importance.csv
- model_outputs\et_selected_10_features_config.json
- model_outputs\et_selected_10_features_best_config.json
- model_outputs\et_selected_10_features_model.pkl
- model_outputs\et_selected_10_features_preprocessor.pkl
- model_outputs\et_selected_10_features_tuning_results.csv
- model_outputs\et_selected_10_features_tuning_runs.csv


## 17. TEST THỬ 1 DÒNG

In [18]:
# ============================================================
# 17. TEST THỬ 1 DÒNG
# ============================================================

sample_index = 110

sample_X = X_test[[sample_index]]
true_value = y_test[sample_index]
predicted_value = model.predict(sample_X)[0]

print("Sample index:", sample_index)
print("Giá trị thật:", true_value)
print("Giá trị dự đoán:", predicted_value)
print("Sai số tuyệt đối:", abs(true_value - predicted_value))

Sample index: 110
Giá trị thật: 66.371
Giá trị dự đoán: 66.52880304948098
Sai số tuyệt đối: 0.15780304948098944


## 18. NHẬN XÉT CHO BÁO CÁO

In [19]:
# ============================================================
# 18. NHẬN XÉT CHO BÁO CÁO
# ============================================================

print("\nNhận xét:")
print("- ExtraTrees được xây dựng từ nhiều cây hồi quy cực kỳ ngẫu nhiên.")
print("- Mỗi node chỉ xét ngẫu nhiên một phần feature và sinh threshold ngẫu nhiên để chia dữ liệu.")
print("- Khác Random Forest, ExtraTrees mặc định không bootstrap mà cho mỗi cây học trên toàn bộ tập train.")
print("- Dự đoán cuối cùng là trung bình dự đoán của toàn bộ các cây.")
print("- Model dùng đúng 10 feature đã chọn, trong đó có feature lịch sử tuổi thọ, Covid, GDP, sanitation và interaction điện-nước.")
print("- is_covid được suy ra từ year nên có liên quan chặt với year; giữ cả hai giúp model tách xu hướng dài hạn và cú sốc Covid.")
print("- MAE nên là metric chính vì đơn vị là năm tuổi thọ.")
print("- RMSE dùng để kiểm tra lỗi lớn; R2 dùng để đánh giá mức độ giải thích biến thiên.")
print("- Nếu train tốt nhưng val/test kém, hãy giảm max_depth hoặc tăng min_samples_leaf.")



Nhận xét:
- ExtraTrees được xây dựng từ nhiều cây hồi quy cực kỳ ngẫu nhiên.
- Mỗi node chỉ xét ngẫu nhiên một phần feature và sinh threshold ngẫu nhiên để chia dữ liệu.
- Khác Random Forest, ExtraTrees mặc định không bootstrap mà cho mỗi cây học trên toàn bộ tập train.
- Dự đoán cuối cùng là trung bình dự đoán của toàn bộ các cây.
- Model dùng đúng 10 feature đã chọn, trong đó có feature lịch sử tuổi thọ, Covid, GDP, sanitation và interaction điện-nước.
- is_covid được suy ra từ year nên có liên quan chặt với year; giữ cả hai giúp model tách xu hướng dài hạn và cú sốc Covid.
- MAE nên là metric chính vì đơn vị là năm tuổi thọ.
- RMSE dùng để kiểm tra lỗi lớn; R2 dùng để đánh giá mức độ giải thích biến thiên.
- Nếu train tốt nhưng val/test kém, hãy giảm max_depth hoặc tăng min_samples_leaf.
